In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

In [ ]:
# Creating embedding model
import os
os.environ['HF_HOME'] = 'D:/huggingface_cache'

embeddings = HuggingFaceEmbeddings(model = "sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [ ]:
documents = [
    Document(page_content="""
    Artificial Intelligence (AI) is transforming industries.
    It is used in healthcare for diagnosis and treatment recommendations.
    AI is also used in finance for fraud detection.
    """),
    
    Document(page_content="""
    Machine learning is a subset of AI.
    It allows systems to learn from data and improve over time.
    Applications include recommendation systems and predictive analytics.
    """),
    
    Document(page_content="""
    Deep learning is a specialized form of machine learning.
    It uses neural networks with many layers.
    It is widely used in image recognition and NLP.
    """),
]

In [ ]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [ ]:
base_retriever = vectorstore.as_retriever()

In [ ]:
compressor = LLMChainExtractor.from_llm(model)

In [ ]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [ ]:
query = "How is AI used in healthcare?"

docs = compression_retriever.get_relevant_documents(query)

for i, doc in enumerate(docs):
    print(f"\nResult {i+1}:")
    print(doc.page_content)